In [ ]:
# This script calculates the change in the average extreme heat season start date and end date between time periods in the CESM ensemble data.
# It can be used to recreate the figure in the manuscript.

In [ ]:
import glob
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import regionmask
from mpl_toolkits.axes_grid1 import make_axes_locatable


In [ ]:
# Indicate location where the CESM Heat Season Characteristics Files are located.
# Script is designed to work with one temperature variable (TMAX or TMIN) at a time.

#OUTPUT_DIR = '/...TMMN/...'
OUTPUT_DIR = '/...TMAX/...'

In [ ]:

# =============================================================================
# 1. DATA LOADING & PREPARATION FUNCTIONS
# =============================================================================
def load_ensemble_data(period_tag):
    """
    Scans the output directory for processed files matching the period tag 
    ('Hist' or 'Fut'), loads them, and stacks them into a single Dataset.
    """
    search_pattern = os.path.join(OUTPUT_DIR, f"ExtremeHeat_*_{period_tag}*.nc")
    file_paths = sorted(glob.glob(search_pattern))
    
    if not file_paths:
        print(f"No files found for '{period_tag}'. Check your directory.")
        return None
        
    print(f"Found {len(file_paths)} files for the {period_tag} period. Stacking...")
    
    ds_list = []
    
    for file_path in file_paths:
        basename = os.path.basename(file_path)
        member_id = basename.split('_')[1] 
        
        ds = xr.open_dataset(file_path)
        ds = ds.expand_dims(member=[member_id])
        ds_list.append(ds)
        
    ds_ensemble = xr.concat(ds_list, dim='member')
    print(f"Successfully created ensemble dataset with dimensions: {dict(ds_ensemble.dims)}")
    
    return ds_ensemble


# The script does not deal with tropical locations (see Methods of the manuscript).
def isolate_extratropics_and_convert_dos(da):
    """
    Masks out the tropics (-23.5 to 23.5) and converts the remaining extra-tropical
    Day of Year (DOY) coordinates to a linear Day of Season (DOS) timeline to prevent
    circular math errors.
    """
    # 1. Mask out tropical latitudes
    da_et = da.where(abs(da.lat) > 23.5)
    
    # 2. Determine approximate start DOY based on hemisphere
    # NH = Month 1 (DOY 0), SH = Month 7 (DOY 180)
    global_start_month = xr.where(da_et.lat >= 0, 1, 7)
    approx_start_doy = (global_start_month - 1) * 30
    
    # 3. Convert DOY to DOS
    # If a day falls before the season boundary, push it into the next cycle (+365.25)
    dos = xr.where(da_et < approx_start_doy, da_et + 365.25, da_et)
    
    return dos


In [ ]:
# =============================================================================
# 2. FDR
# =============================================================================
def apply_fdr(p_values, alpha=0.05):
    """
    Applies the Benjamini-Hochberg False Discovery Rate (FDR) procedure.
    """
    p_flat = p_values.values.flatten()
    valid_idx = ~np.isnan(p_flat)
    p_valid = p_flat[valid_idx]
    
    # Sort p-values
    sorted_idx = np.argsort(p_valid)
    p_sorted = p_valid[sorted_idx]
    
    # Calculate critical values
    m = len(p_sorted)
    q_values = (np.arange(1, m + 1) / m) * alpha
    
    # Find the largest p-value that is less than its critical value
    significant = p_sorted <= q_values
    if np.any(significant):
        max_idx = np.where(significant)[0][-1]
        p_threshold = p_sorted[max_idx]
    else:
        p_threshold = -1.0 # No significant pixels
        
    # Create the spatial mask
    return p_values <= p_threshold



In [ ]:
def calculate_emergence_robustness(da_hist, da_fut, agreement_threshold=0.666, n_permutations=1000):
    """
    Calculates significance on a member-by-member basis (shuffling years).
    A grid cell is robust if >= threshold (e.g., 66%) of members show a 
    statistically significant shift in the SAME direction as the ensemble mean.
    """
    print("1. Calculating Ensemble Mean Change & Direction...")
    # Calculate the mean start/end date across years for each member
    mem_mean_hist = da_hist.mean(dim='year', skipna=True)
    mem_mean_fut  = da_fut.mean(dim='year', skipna=True)
    
    # Calculate difference per member, then the overall ensemble mean shift
    mem_diff = mem_mean_fut - mem_mean_hist
    ens_mean_diff = mem_diff.mean(dim='member', skipna=True)
    
    # Get the overall direction of the ensemble mean change
    ens_sign = np.sign(ens_mean_diff.values)
    
    total_members = da_hist.sizes['member']
    n_years_hist = da_hist.sizes['year']
    n_years_total = n_years_hist + da_fut.sizes['year']
    
    # Initialize a blank map to count how many members pass the test
    robust_member_count = xr.zeros_like(ens_mean_diff)
    
    print(f"2. Running Member-by-Member Permutation Tests ({total_members} members, {n_permutations} shuffles each)...")
    np.random.seed(42)
    
    for m in range(total_members):
        print(f"   Processing Member {m+1}/{total_members}...", end='\r')
        
        # Extract data for just this single member as pure numpy arrays for speed
        m_hist = da_hist.isel(member=m).values
        m_fut  = da_fut.isel(member=m).values
        m_obs_diff = mem_diff.isel(member=m).values
        
        # Get the direction of THIS member's change
        m_sign = np.sign(m_obs_diff)
        
        # Combine the 60 years for shuffling (axis=0 is the year dimension)
        combined = np.concatenate([m_hist, m_fut], axis=0) 
        exceed_count = np.zeros_like(m_obs_diff)
        
        # Shuffle years for this specific member
        for _ in range(n_permutations):
            idx = np.random.permutation(n_years_total)
            shuffled = combined[idx, ...]
            
            # Using nanmean because start/end dates are averages
            pseudo_hist = np.nanmean(shuffled[:n_years_hist, ...], axis=0)
            pseudo_fut  = np.nanmean(shuffled[n_years_hist:, ...], axis=0)
            pseudo_diff = pseudo_fut - pseudo_hist
            
            exceed_count += (np.abs(pseudo_diff) >= np.abs(m_obs_diff))
            
        # Calculate p-values and apply FDR for THIS member only
        p_values = exceed_count / n_permutations
        p_val_da = xr.DataArray(p_values, coords=ens_mean_diff.coords, dims=ens_mean_diff.dims)
        m_fdr_mask = apply_fdr(p_val_da, alpha=0.05)
        
        # Check criteria: Is it significant AND does the sign match the ensemble average?
        sign_match = (m_sign == ens_sign)
        member_is_robust = m_fdr_mask & sign_match
        
        # Add the passing grid cells to our running tally
        robust_member_count += member_is_robust
        
    print(f"\n3. Applying {agreement_threshold*100:.1f}% Agreement Threshold...")
    final_emergence_mask = (robust_member_count / total_members) >= agreement_threshold
    
    return ens_mean_diff, final_emergence_mask

In [ ]:

# =============================================================================
# 3. PLOTTING
# =============================================================================
def plot_robust_map_with_zonal(da_hist, da_fut, ens_diff, robust_mask, title, 
                               label='Shift in Days', cmap='RdBu_r', 
                               levels=None, xlim=None):
    
    print("1. Realigning Longitudes (0-360 to -180-180)...")
    def shift_lon(da):
        return da.assign_coords(lon=(((da.lon + 180) % 360) - 180)).sortby('lon')
        
    da_hist = shift_lon(da_hist)
    da_fut = shift_lon(da_fut)
    ens_diff = shift_lon(ens_diff)
    robust_mask = shift_lon(robust_mask)

    print("2. Applying Land Mask...")
    land_mask = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(da_hist)
    
    da_hist_land = da_hist.where(land_mask == 0)
    da_fut_land = da_fut.where(land_mask == 0)
    ens_diff_land = ens_diff.where(land_mask == 0)
    
    print("3. Calculating Zonal Means and Full Ensemble Spread (Land Only)...")
    mem_mean_hist = da_hist_land.mean(dim='year', skipna=True)
    mem_mean_fut  = da_fut_land.mean(dim='year', skipna=True)
    
    mem_diff = mem_mean_fut - mem_mean_hist
    zonal_mem_diff = mem_diff.mean(dim='lon', skipna=True)
    
    zonal_ens_mean = zonal_mem_diff.mean(dim='member', skipna=True)
    zonal_ens_min = zonal_mem_diff.min(dim='member', skipna=True)
    zonal_ens_max = zonal_mem_diff.max(dim='member', skipna=True)
    lats = zonal_ens_mean.lat

    print("4. Generating Perfectly Aligned Figure...")
    fig, ax_map = plt.subplots(figsize=(12, 7), subplot_kw={'projection': ccrs.PlateCarree()})
    ax_map.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())
    
    divider = make_axes_locatable(ax_map)
    ax_zonal = divider.append_axes("right", size="20%", pad=0.3, axes_class=plt.Axes)
    cax = divider.append_axes("bottom", size="5%", pad=0.4, axes_class=plt.Axes)

    # ==========================================
    # LEFT: MASKED MAP PLOT
    # ==========================================
    robust_diff = ens_diff_land.where(robust_mask)
    
    # explicitly plot tropical land as gray because we are masking it out here
    tropical_band = xr.where(abs(da_hist.lat) <= 23.5, 1, np.nan)
    tropical_land = tropical_band.where(land_mask == 0)
    
    tropical_land.plot(
        ax=ax_map,
        transform=ccrs.PlateCarree(),
        cmap=mcolors.ListedColormap(['lightgray']),
        add_colorbar=False,
        zorder=1
    )
    # ---------------------------------------------------

    map_plot = robust_diff.plot(
        ax=ax_map, 
        transform=ccrs.PlateCarree(),
        cmap=cmap, 
        levels=levels, 
        extend='both',
        add_colorbar=False,
        zorder=1
    )


    ax_map.add_feature(cfeature.OCEAN, facecolor='white', edgecolor='none', zorder=2)
    ax_map.coastlines(color='black', linewidth=0.8, zorder=3)
    ax_map.add_feature(cfeature.COASTLINE, linestyle=':', edgecolor='gray', linewidth=0.5, zorder=3)
    
    gl = ax_map.gridlines(
        draw_labels=True, 
        xlocs=np.arange(-180, 181, 60), 
        ylocs=np.arange(-90, 91, 30), 
        color='lightgray', 
        linewidth=0.8, 
        linestyle='-', 
        zorder=4
    )
    gl.top_labels = False
    gl.right_labels = False
    
    ax_map.set_title(title, fontsize=14, fontweight='bold', pad=15)
    
    cbar = plt.colorbar(map_plot, cax=cax, orientation='horizontal')
    cbar.set_label(label, fontsize=12)
    
    # RIGHT: ZONAL MEAN PLOT
    ax_zonal.axvline(0, color='black', linestyle='-', linewidth=1, zorder=2)
    
    ax_zonal.fill_betweenx(
        lats, 
        zonal_ens_min, 
        zonal_ens_max, 
        color='gray', alpha=0.3, zorder=1    )
      
    ax_zonal.plot(zonal_ens_mean, lats, color='red', linewidth=2, zorder=3)
    
    ax_zonal.set_ylim(-90, 90) 
    ax_zonal.set_xlim(xlim[0], xlim[1])
    ax_zonal.set_yticks(np.arange(-90, 100, 30))
    ax_zonal.set_yticklabels(['90°S', '60°S', '30°S', 'EQ', '30°N', '60°N', '90°N'])
    ax_zonal.yaxis.tick_right()
    
    ax_zonal.set_xlabel(label)
    ax_zonal.set_title('Land-Only Zonal Mean', fontsize=12, fontweight='bold')
    ax_zonal.grid(True, linestyle=':', alpha=0.6)
    
    plt.show()
    return fig


In [ ]:
import matplotlib.colors as mcolors



# =============================================================================
# 4. MAIN EXECUTION
# =============================================================================
print("--- Loading Historical Ensemble ---")
ens_hist_raw = load_ensemble_data("Hist_1966_1995")

print("\n--- Loading Future Ensemble ---")
ens_fut_raw = load_ensemble_data("Fut_2026_2055") # change depending on whether you want the 1996-2025 or 2026-2055 period.

if ens_hist_raw is not None and ens_fut_raw is not None:
    # --- PROCESS START DATES ---
    print("\n" + "="*40)
    print("ANALYZING START DATE SHIFTS (Extra-Tropics Only)")
    print("="*40)
    
    dos_start_hist = isolate_extratropics_and_convert_dos(ens_hist_raw['s1_start'])
    dos_start_fut  = isolate_extratropics_and_convert_dos(ens_fut_raw['s1_start'])
    
    # UPDATED FUNCTION CALL
    diff_start, final_mask_start = calculate_emergence_robustness(
        dos_start_hist, 
        dos_start_fut, 
        agreement_threshold=0.666, # 2/3 agreement for emergence
        n_permutations=1000
    )
    
    fig_start = plot_robust_map_with_zonal(
        da_hist=dos_start_hist,
        da_fut=dos_start_fut,
        ens_diff=diff_start, 
        robust_mask=final_mask_start,
        #title="Avg CESM Change in Extra-Tropical TMMN Season Start (2026-2055) - (1966-1995)",
        title="Avg CESM Change in Extra-Tropical TMAX Season Start (2026-2055) - (1966-1995)",
        label="Days",
        cmap="RdBu", 
        levels=np.arange(-40, 45, 5),
        xlim=(-90, 90)                 
    )
    #fig_start.savefig("plots/CESM_TMIN_Future_26-55_SeasonStart_Change.pdf", format="pdf", bbox_inches="tight")
    fig_start.savefig("plots/CESM_TMAX_Future_26-55_SeasonStart_Change.pdf", format="pdf", bbox_inches="tight")

    # --- PROCESS END DATES ---
    print("\n" + "="*40)
    print("ANALYZING END DATE SHIFTS (Extra-Tropics Only)")
    print("="*40)
    
    dos_end_hist = isolate_extratropics_and_convert_dos(ens_hist_raw['s1_end'])
    dos_end_fut  = isolate_extratropics_and_convert_dos(ens_fut_raw['s1_end'])
    
    # FUNCTION CALL
    diff_end, final_mask_end = calculate_emergence_robustness(
        dos_end_hist, 
        dos_end_fut, 
        agreement_threshold=0.666, # 2/3 agreement for emergence
        n_permutations=1000
    )
    
    fig_end = plot_robust_map_with_zonal(
        da_hist=dos_end_hist,
        da_fut=dos_end_fut,
        ens_diff=diff_end, 
        robust_mask=final_mask_end,
        #title="Avg CESM Change in Extra-Tropical TMMN Season End (2026-2055) - (1966-1995)",
        title="Avg CESM Change in Extra-Tropical TMAX Season End (2026-2055) - (1966-1995)",
        levels=np.arange(-40, 45, 5),
        label="Days",
        cmap="RdBu_r", 
        xlim=(-90, 90)                 
    )
    #fig_end.savefig("plots/CESM_TMIN_Future_26-55_SeasonEnd_Change.pdf", format="pdf", bbox_inches="tight")
    #fig_end.savefig("plots/CESM_TMAX_Future_26-55_SeasonEnd_Change.pdf", format="pdf", bbox_inches="tight")